<a href="https://colab.research.google.com/drive/1E5jd0HhCa7ows1vdmtr9xyMz9LDZZ9bL">Abre este cuaderno en Google Colab</a>

# Preparación de Datos

Este documento detalla las metodologías y pasos fundamentales para depurar, transformar y estructurar la información bruta antes de alimentar un modelo de Machine Learning.

## 1. Información del Dataset

### Contexto y Origen
Trabajaremos con **NSL-KDD**, un conjunto de datos creado para subsanar los defectos identificados en el histórico dataset KDD'99. Si bien no refleja a la perfección la enorme complejidad de las redes modernas, su uso está ampliamente justificado en la comunidad académica porque permite comparar métodos de detección de intrusos de forma estandarizada.

Al tener un tamaño de registros equilibrado, es posible procesarlo en su totalidad sin tener que recurrir a muestreos aleatorios, garantizando que las métricas obtenidas sean reproducibles y consistentes.

### Archivos Disponibles en el Repositorio
* **KDDTrain+.ARFF**: Datos de entrenamiento completos con etiquetas binarias (formato ARFF).
* **KDDTrain+.TXT**: Datos de entrenamiento completos que incluyen la categoría específica del ataque y su nivel de dificultad (formato CSV).
* **Versiones reducidas**: Archivos terminados en `_20Percent` para pruebas de concepto más veloces.
* **KDDTest+.ARFF / TXT**: Conjuntos destinados exclusivamente a la evaluación final del modelo.

### Enlace de Obtención
Puedes descargar los ficheros originales aquí: https://iscxdownloads.cs.unb.ca/iscxdownloads/NSL-KDD/#NSL-KDD


## 2. Importación de Herramientas

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

## 3. Definición de Rutinas Auxiliares
Establecemos los encabezados manuales del dataset, ya que los archivos TXT carecen de nombres de columna.

In [16]:
NOMBRES_COLUMNAS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "class", "difficulty"
]

def cargar_datos_kdd(ruta_archivo):
    """Lee el dataset NSL-KDD y le asigna los nombres estructurales correctos."""
    return pd.read_csv(ruta_archivo, header=None, names=NOMBRES_COLUMNAS)

In [17]:
def particionar_datos(df, semilla=42, mezclar=True, estratificar=None):
    """Divide el DataFrame en Entrenamiento (60%), Validación (20%) y Prueba (20%)."""
    estrato = df[estratificar] if estratificar else None
    entrenamiento, prueba = train_test_split(
        df, test_size=0.4, random_state=semilla, shuffle=mezclar, stratify=estrato)
    
    estrato_prueba = prueba[estratificar] if estratificar else None
    validacion, prueba = train_test_split(
        prueba, test_size=0.5, random_state=semilla, shuffle=mezclar, stratify=estrato_prueba)
    
    return entrenamiento, validacion, prueba

## 4. Ingesta Inicial y Exploración

In [18]:
df = cargar_datos_kdd("KDDTrain+.txt")
df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## 5. Segmentación Estratificada
Aplicamos la función auxiliar para separar los datos asegurando que la proporción del tipo de protocolo se mantenga uniforme en los tres bloques generados.

In [19]:
train_set, val_set, test_set = particionar_datos(df, estratificar='protocol_type')

In [20]:
print(f"Volumen de Entrenamiento: {len(train_set)}")
print(f"Volumen de Validación: {len(val_set)}")
print(f"Volumen de Prueba: {len(test_set)}")

Volumen de Entrenamiento: 75583
Volumen de Validación: 25195
Volumen de Prueba: 25195


## 6. Gestión de Valores Inexistentes (Imputación)
En escenarios del mundo real, los conjuntos de datos suelen venir con casillas vacías o corruptas (NaN). Para solucionarlo sin perder registros valiosos, emplearemos `SimpleImputer` para rellenar estos vacíos utilizando la mediana estadística de cada variable numérica.

In [21]:
# Filtramos temporalmente solo las columnas con valores numéricos
X_train_num = train_set.select_dtypes(include=[np.number])

# Inicializamos y ajustamos el imputador
imputador = SimpleImputer(strategy='median')
imputador.fit(X_train_num)

# Ejecutamos la transformación para cubrir los huecos detectados
X_train_num_limpio = imputador.transform(X_train_num)

# Reconstruimos el DataFrame con la data corregida
X_train_num_df = pd.DataFrame(X_train_num_limpio, columns=X_train_num.columns, index=X_train_num.index)
X_train_num_df.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0.0,407.0,53508.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.11,0.03,0.0,0.0,0.0,0.0,21.0
31899,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.02,0.05,0.00,0.00,1.0,1.0,0.0,0.0,21.0
108116,0.0,304.0,636.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.03,0.06,0.0,0.0,0.0,0.0,21.0
89913,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.0,0.06,0.07,0.00,0.00,1.0,1.0,0.0,0.0,21.0
106319,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,1.00,0.00,1.00,0.57,0.0,0.0,0.0,0.0,16.0


## 7. Codificación de Atributos Categóricos
Las redes neuronales y los clasificadores matemáticos requieren que todas las entradas sean números. 
Emplearemos `OneHotEncoder` para transformar variables de texto (como el tipo de ataque o el protocolo) en matrices de ceros y unos. Configuramos `handle_unknown='ignore'` para evitar caídas del sistema si en la fase de pruebas ingresa una categoría nunca antes vista.

In [22]:
codificador_oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

### Uso de Pandas: `get_dummies`
Una alternativa sumamente rápida y directa para hacer One-Hot Encoding directamente sobre un DataFrame es utilizar la función integrada de Pandas.

In [23]:
# Aislamos el texto y generamos las columnas binarias
X_train_cat = train_set.select_dtypes(include=['object'])
X_train_cat_dummies = pd.get_dummies(X_train_cat)

X_train_cat_dummies.head()

/tmp/ipykernel_14263/2953257045.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train_cat = train_set.select_dtypes(include=['object'])


,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_aol,service_auth,service_bgp,service_courier,...,class_phf,class_pod,class_portsweep,class_rootkit,class_satan,class_smurf,class_spy,class_teardrop,class_warezclient,class_warezmaster
113467,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
31899,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
108116,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
89913,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
106319,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 8. Normalización y Escalado de Características
Cuando las columnas tienen magnitudes muy dispares (por ejemplo, recuento de bytes frente a tasas de error en porcentaje), el modelo puede otorgar falsa prioridad a los números grandes. 

Implementamos `RobustScaler`, una técnica de escalado numérico que es especialmente efectiva mitigando la influencia negativa de valores extremos (outliers).

In [24]:
escalador = RobustScaler()
X_train_escalado = escalador.fit_transform(X_train_num_df)

X_train_escalado_df = pd.DataFrame(X_train_escalado, columns=X_train_num_df.columns, index=X_train_num_df.index)
X_train_escalado_df.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0.0,1.324818,101.920000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,1.833333,1.5,0.0,0.0,0.0,0.0,0.333333
31899,0.0,-0.160584,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.236735,-0.515789,0.285714,0.000000,0.0,1.0,1.0,0.0,0.0,0.333333
108116,0.0,0.948905,1.211429,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.787755,0.515789,-0.428571,0.500000,3.0,0.0,0.0,0.0,0.0,0.333333
89913,0.0,-0.160584,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.191837,-0.473684,0.571429,0.000000,0.0,1.0,1.0,0.0,0.0,0.333333
106319,0.0,-0.131387,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.224490,0.515789,-0.428571,16.666667,28.5,0.0,0.0,0.0,0.0,-1.333333


## Conclusión del Proceso
Tras completar este cuaderno, hemos logrado los siguientes hitos de preparación:

1. **Imputación Estratégica**: Las filas con fallos y vacíos fueron rellenadas mediante medianas para evitar pérdida de datos.
2. **Vectorización Segura**: Se transformó la información verbal a representaciones lógicas ortogonales (One-Hot).
3. **Ecualización de Escalas**: Todas las columnas operan ahora bajo un rango numérico armonizado y blindado contra anomalías.

El conjunto de datos se encuentra finalmente en óptimas condiciones para iniciar la fase de diseño arquitectónico y entrenamiento algorítmico.